<a href="https://colab.research.google.com/github/Aranzazu21/Machine-Learning/blob/main/P3P2_Regresi%C3%B3n_Log%C3%ADstica.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score
from google.colab import drive
import warnings
warnings.filterwarnings('ignore')

drive.mount('/content/drive')


ruta = '/content/drive/MyDrive/Ciencia de datos /dataset_limpio_transformado (2).xlsx'
df = pd.read_excel(ruta)

df['Fecha_DT'] = pd.to_datetime(df['Fecha_DT'])
df = df.sort_values('Fecha_DT').reset_index(drop=True)

df['alcanzo_meta_semanal'] = np.where(df['Peso_Kg'].shift(1) - df['Peso_Kg'] >= 0.5, 1, 0)
df['alcanzo_meta_semanal'] = df['alcanzo_meta_semanal'].fillna(0)

df['carga_semanal_km'] = df['carga_semanal_km'].fillna(0)
df['conteo_pasos'] = df['conteo_pasos'].fillna(0)
df['tiempo_actividad_min'] = df['tiempo_actividad_min'].fillna(0)

X = df[['conteo_pasos', 'carga_semanal_km', 'tiempo_actividad_min']]
y = df['alcanzo_meta_semanal']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

modelo_logistico = LogisticRegression()
modelo_logistico.fit(X_train, y_train)

# generar predicciones y probabilidades
y_pred = modelo_logistico.predict(X_test)
y_prob = modelo_logistico.predict_proba(X_test)[:, 1]

# extraer coeficientes y calcular odds ratios
b0 = modelo_logistico.intercept_[0]
coefs = modelo_logistico.coef_[0]
odds_ratios = np.exp(coefs)

print("coeficientes y odds ratios:")
print(f"intercepto b0: {b0:.4f}")
print(f"pasos b1: {coefs[0]:.6f} | odds ratio: {odds_ratios[0]:.6f}")
print(f"carga km b2: {coefs[1]:.6f} | odds ratio: {odds_ratios[1]:.6f}")
print(f"tiempo min b3: {coefs[2]:.6f} | odds ratio: {odds_ratios[2]:.6f}")

# matriz de confusion
print("\nmatriz de confusion y metricas:")
cm = confusion_matrix(y_test, y_pred)
print(f"verdaderos negativos vn: {cm[0,0]}")
print(f"falsos positivos fp: {cm[0,1]}")
print(f"falsos negativos fn: {cm[1,0]}")
print(f"verdaderos positivos vp: {cm[1,1]}")

print("\nreporte de clasificacion:")
print(classification_report(y_test, y_pred))

# calcular auc roc
auc = roc_auc_score(y_test, y_prob)
print(f"auc-roc score: {auc:.4f}")


print("\nevaluacion de perfiles:")
perfiles = pd.DataFrame({
    'conteo_pasos': [15000, 3000],
    'carga_semanal_km': [100, 0],
    'tiempo_actividad_min': [300, 30]
})
probs = modelo_logistico.predict_proba(perfiles)[:, 1]
preds = modelo_logistico.predict(perfiles)

print(f"perfil 1 (alta actividad) -> probabilidad p(y=1): {probs[0]*100:.2f}% | decision final: {preds[0]}")
print(f"perfil 2 (baja actividad) -> probabilidad p(y=1): {probs[1]*100:.2f}% | decision final: {preds[1]}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
coeficientes y odds ratios:
intercepto b0: -0.1644
pasos b1: 0.000000 | odds ratio: 1.000000
carga km b2: -0.008433 | odds ratio: 0.991602
tiempo min b3: 0.000000 | odds ratio: 1.000000

matriz de confusion y metricas:
verdaderos negativos vn: 4
falsos positivos fp: 0
falsos negativos fn: 1
verdaderos positivos vp: 0

reporte de clasificacion:
              precision    recall  f1-score   support

           0       0.80      1.00      0.89         4
           1       0.00      0.00      0.00         1

    accuracy                           0.80         5
   macro avg       0.40      0.50      0.44         5
weighted avg       0.64      0.80      0.71         5

auc-roc score: 0.5000

evaluacion de perfiles:
perfil 1 (alta actividad) -> probabilidad p(y=1): 26.74% | decision final: 0
perfil 2 (baja actividad) -> probabilidad p(y=1): 45.90% | decision final: